# Entrenar tu propia voz Piper para LoudVoxFine-tuning del checkpoint español oficial con TUS grabaciones. Requiere: **GPU activada** (Entorno de ejecución → Cambiar tipo de entorno → T4 GPU) y tu `dataset.zip` (ver README de la carpeta colab/).Tiempo estimado: 2-4 horas. El resultado (`mi_voz.onnx` + `.json`) corre local en tu PC.

In [ ]:
#@title 1. Verificar GPUimport torchassert torch.cuda.is_available(), "Activá la GPU: Entorno de ejecución -> Cambiar tipo -> T4 GPU"print("GPU OK:", torch.cuda.get_device_name(0))

In [ ]:
#@title 2. Instalar Piper (entrenamiento)%cd /content!git clone -q https://github.com/rhasspy/piper.git%cd /content/piper/src/python!pip install -q -e .!pip install -q "pytorch-lightning~=1.9" "onnxruntime" "espeak-phonemizer" "librosa" "numpy<2"!apt-get install -yq espeak-ng > /dev/null!bash build_monotonic_align.shprint("Piper instalado")

In [ ]:
#@title 3. Subir tu dataset.zip (wavs/ + metadata.csv)from google.colab import filesimport zipfile, osup = files.upload()  # elegí dataset.zipname = list(up.keys())[0]os.makedirs("/content/dataset", exist_ok=True)with zipfile.ZipFile(name) as z: z.extractall("/content/dataset_raw")# aceptar zip con o sin carpeta contenedoraroot = "/content/dataset_raw"while not os.path.exists(os.path.join(root, "metadata.csv")):    subs = [d for d in os.listdir(root) if os.path.isdir(os.path.join(root, d))]    assert subs, "No encuentro metadata.csv en el zip"    root = os.path.join(root, subs[0])print("Dataset en:", root)n = sum(1 for _ in open(os.path.join(root, "metadata.csv"), encoding="utf-8"))print(f"{n} frases")DATASET_DIR = root

In [ ]:
#@title 4. Preprocesar (texto -> fonemas, audio -> espectrogramas)%cd /content/piper/src/python!python -m piper_train.preprocess \  --language es \  --input-dir "{DATASET_DIR}" \  --output-dir /content/train_out \  --dataset-format ljspeech \  --single-speaker \  --sample-rate 22050print("Preprocesado listo")

In [ ]:
#@title 5. Descargar el checkpoint base español (punto de partida)!wget -q -O /content/base_es.ckpt "https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/es/es_ES/davefx/medium/epoch%3D2218-step%3D562840.ckpt"import os; print("Checkpoint:", os.path.getsize("/content/base_es.ckpt")//1_000_000, "MB")

In [ ]:
#@title 6. Entrenar (fine-tuning). 2-4 h; podés cortar antes y probar.%cd /content/piper/src/python!python -m piper_train \  --dataset-dir /content/train_out \  --accelerator gpu --devices 1 \  --batch-size 16 \  --validation-split 0.0 \  --num-test-examples 0 \  --max_epochs 2600 \  --resume_from_checkpoint /content/base_es.ckpt \  --checkpoint-epochs 5 \  --precision 32 \  --quality medium

In [ ]:
#@title 7. Exportar a .onnx y descargarimport glob, shutil, osckpts = sorted(glob.glob("/content/train_out/lightning_logs/*/checkpoints/*.ckpt"), key=os.path.getmtime)assert ckpts, "No hay checkpoints todavía (corré la celda 6 al menos unos epochs)"last = ckpts[-1]print("Exportando:", last)%cd /content/piper/src/python!python -m piper_train.export_onnx "{last}" /content/mi_voz.onnxshutil.copy("/content/train_out/config.json", "/content/mi_voz.onnx.json")from google.colab import filesfiles.download("/content/mi_voz.onnx")files.download("/content/mi_voz.onnx.json")print("Copiá ambos archivos a tu carpeta de voces de LoudVox")